# Seminar 1 (part 2) — NumPy crash-course for ML

**Goal:** master exactly the NumPy you need for Seminar 2 (linear regression with two features): indexing, sums over axes, broadcasting, and scalar (dot) products.

**How to work:** run top-to-bottom (`Shift + Enter`), fill every ✏️ **TODO / YOUR CODE HERE**. Self-check asserts tell you if you are right.

**Plan:**
1. Arrays: creation, shape, dtype
2. Indexing, slicing, views vs copies
3. Boolean masks and fancy indexing (filtering rows)
4. Sums and means along axes (`axis=0 / axis=1`)
5. Broadcasting: rules, row/column vectors, standardisation
6. Scalar (dot) products: `*` vs `dot` vs `@`, norms, cosine
7. Random numbers with seed
8. Plotting from NumPy arrays
9. Mini-project: $\hat y = b + w_1 x_1 + w_2 x_2$ and MSE, NumPy-only
10. Final checklist

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print("numpy", np.__version__)

## 1. Arrays: creation, shape, dtype

A NumPy array is a grid of numbers of one type. `shape` is its size: `(n,)` = $n$ numbers, `(n, 2)` = $n$ rows × 2 columns. `dtype` is the number type (usually `float64`).

In [ ]:
a = np.array([1.0, 2.0, 3.0])
print("a =", a, " shape =", a.shape, " dtype =", a.dtype)

X = np.array([[1.0, 10.0],
              [2.0, 20.0],
              [3.0, 30.0],
              [4.0, 40.0]])
print("X shape =", X.shape, " (4 rows, 2 columns)")
print(X)
print("zeros:", np.zeros(3), " ones:", np.ones((2, 2)), " range:", np.arange(5))

In [ ]:
# ✏️ TODO 1: build a column [10, 20, 30] and a 2x3 grid [[1,2,3],[4,5,6]]. Check shapes.
col = np.array([10.0, 20.0, 30.0])  # <-- YOUR CODE HERE
grid = np.array([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])  # <-- YOUR CODE HERE
print(col.shape, grid.shape)
assert col.shape == (3,) and grid.shape == (2, 3)
print("Task 1 OK ✓")

## 2. Indexing and slicing (read slowly — this is the core skill)

Counting starts at 0. For a grid `X` with shape `(n, 2)`:

- `X[0]` → first **row**; `X[-1]` → last row;
- `X[:, 0]` → first **column** (all rows, column 0); `X[:, 1]` → second column;
- `X[0, 1]` → single cell (row 0, column 1);
- `X[1:3]` → rows 1..2 (right end excluded); `X[::2]` → every second row; `X[:, 0:1]` → first column **kept as a grid** with shape `(n, 1)`;
- `X.T` → transposed grid (rows ↔ columns), shape `(2, n)`.

⚠️ **Views vs copies:** a plain slice (`X[1:3]`, `X[:, 0]`) is a *view* — it shares memory with `X`, so writing into it changes `X`. Use `.copy()` when you need an independent array.

In [ ]:
print("first row X[0] =", X[0])
print("last row X[-1] =", X[-1])
print("first column X[:, 0] =", X[:, 0], " shape:", X[:, 0].shape)
print("first column as grid X[:, 0:1].shape =", X[:, 0:1].shape)
print("cell X[0, 1] =", X[0, 1])
print("middle rows X[1:3]:\n", X[1:3])
print("every second row X[::2]:\n", X[::2])
print("transposed X.T shape:", X.T.shape, "\n", X.T)

# View vs copy demo
v = X[:2]          # view into X
c = X[:2].copy()   # independent copy
print("\nview shares memory:", np.shares_memory(v, X), "| copy shares memory:", np.shares_memory(c, X))

In [ ]:
# ✏️ TODO 2: slicing practice on X (4x2 grid from above).
last_col = X[:, 1]  # <-- YOUR CODE HERE (second column, shape (4,))
middle = X[1:3, :]  # <-- YOUR CODE HERE (rows 1 and 2, all columns)
odd_rows = X[1::2, :]  # <-- YOUR CODE HERE (rows 1 and 3)
print("last_col:", last_col, " shape:", last_col.shape)
print("middle:\n", middle)
print("odd_rows:\n", odd_rows)
assert np.allclose(last_col, [10, 20, 30, 40])
assert np.allclose(middle, [[2, 20], [3, 30]])
assert np.allclose(odd_rows, [[2, 20], [4, 40]])
print("Task 2 OK ✓")

# ✏️ TODO 2b (views!): fix the bug — make an INDEPENDENT copy before modifying.
buggy = X[:2].copy()  # <-- YOUR CODE HERE (was X[:2], which would corrupt X!)
buggy[0, 0] = 999.0
assert X[0, 0] == 1.0, "X got corrupted — you forgot .copy()!"
print("View/copy check OK ✓  (X is intact)")

## 3. Boolean masks and fancy indexing (filtering rows)

A comparison like `X[:, 0] > 2` gives a mask of `True/False`. Put it in brackets — `X[mask]` keeps only matching rows. A list of row numbers `X[[0, 3]]` picks rows 0 and 3 (*fancy indexing*, always a copy). `np.where`, `argmax`, `argmin` answer "where is...?".

In [ ]:
mask = X[:, 0] > 2.0
print("mask:", mask)
print("rows with x1 > 2:\n", X[mask])
print("rows 0 and 3:\n", X[[0, 3]])
y = np.array([3.0, 5.0, 7.0, 9.0])
print("where y > 6:", np.where(y > 6.0)[0])
print("argmax(y):", np.argmax(y), " argmin(y):", np.argmin(y))

In [ ]:
# ✏️ TODO 3: keep rows where the second column x2 >= 30, then take their y values.
sel = X[X[:, 1] >= 30.0]  # <-- YOUR CODE HERE (filtered rows)
sel_y = y[X[:, 1] >= 30.0]  # <-- YOUR CODE HERE (matching y values)
print(sel, sel_y)
assert np.allclose(sel, [[3, 30], [4, 40]]) and np.allclose(sel_y, [7, 9])
print("Task 3 OK ✓")

## 4. Sums and means along axes

`axis=0` collapses **rows** → one value per column; `axis=1` collapses **columns** → one value per row; no `axis` → single number over everything. `keepdims=True` keeps the result as a row/grid so it broadcasts nicely later (Section 5).

In [ ]:
print("all: sum =", np.sum(X), " mean =", np.mean(X))
print("per column (axis=0): sum =", np.sum(X, axis=0), " mean =", np.mean(X, axis=0))
print("per row (axis=1): sum =", np.sum(X, axis=1))
print("per column min:", np.min(X, axis=0), " max:", np.max(X, axis=0), " std:", np.std(X, axis=0))
print("keepdims mean shape:", np.mean(X, axis=0, keepdims=True).shape)

In [ ]:
# ✏️ TODO 4: the 4 sums Seminar 2 builds its system of equations from.
s1 = np.sum(X[:, 0])  # <-- YOUR CODE HERE (sum of x1)
s2 = np.sum(X[:, 1])  # <-- YOUR CODE HERE (sum of x2)
s12 = np.sum(X[:, 0] * X[:, 1])  # <-- YOUR CODE HERE (sum of x1*x2)
s1y = np.sum(X[:, 0] * y)  # <-- YOUR CODE HERE (sum of x1*y)
print(s1, s2, s12, s1y)
assert (s1, s2, s12, s1y) == (10.0, 100.0, 300.0, 70.0)
# same sums via axis=0 in one call:
assert np.allclose(np.sum(X, axis=0), [s1, s2])
print("Task 4 OK ✓")

## 5. Broadcasting: one formula for all rows

NumPy stretches small shapes to big ones automatically. Rules (from the right):

1. a scalar stretches to everything: `X + 1` adds 1 to each cell;
2. a row of shape `(2,)` stretches over rows: `(4, 2) + (2,)` works — each row gets the same addition;
3. shapes `(4, 1)` and `(4, 2)` stretch along columns; `(1, 2)` stretches along rows;
4. `(4,)` vs `(4, 1)` are NOT the same: the first is a flat column, the second an explicit grid column — reshape with `[:, None]` or `keepdims` when a formula needs a column.

Main use in this course — standardisation (zero mean, unit variance) in one line:
`Xs = (X - mu) / sigma` with `mu`, `sigma` of shape `(2,)` computed on train only.

In [ ]:
print("scalar:\n", X + 100.0)                    # (4,2) + scalar
print("row vector:\n", X + np.array([1.0, 2.0]))  # (4,2) + (2,): per-column shift
print("per-row scale:\n", X * np.array([[1.0], [2.0], [3.0], [4.0]]))  # (4,2) * (4,1)

# Standardisation via broadcasting
mu = np.mean(X, axis=0)       # shape (2,)
sigma = np.std(X, axis=0)     # shape (2,)
Xs = (X - mu) / sigma
print("mu:", mu, " sigma:", sigma)
print("standardised:\n", np.round(Xs, 3))
print("check: mean ≈", np.mean(Xs, axis=0), " std ≈", np.std(Xs, axis=0))

In [ ]:
# ✏️ TODO 5: standardise X yourself and prove (n,) vs (n,1) matters.
mu = np.mean(X, axis=0)  # <-- YOUR CODE HERE, shape must be (2,)
sigma = np.std(X, axis=0)  # <-- YOUR CODE HERE, shape must be (2,)
Z = (X - mu) / sigma  # <-- YOUR CODE HERE (broadcasting!)
assert mu.shape == (2,) and sigma.shape == (2,) and Z.shape == (4, 2)
assert np.allclose(np.mean(Z, axis=0), [0, 0], atol=1e-12)
assert np.allclose(np.std(Z, axis=0), [1, 1])
flat, col = X[:, 0], X[:, 0:1]
print("flat shape:", flat.shape, "| grid-column shape:", col.shape)
assert flat.shape == (4,) and col.shape == (4, 1)
assert np.allclose(col[:, 0], flat)  # same numbers, different shapes
print("Task 5 OK ✓")

## 6. Scalar (dot) products: `*` vs `dot` vs `@`

- `a * b` = **elementwise**: each cell multiplied separately (needs equal shapes after broadcasting);
- `a.dot(b)` / `a @ b` = **scalar (dot) product**: multiply pairwise and add up — one number from two columns: $a \cdot b = \sum_k a_k b_k$;
- for grids: `(n, 2) @ (2,)` → `(n,)` (each row dotted with the weights — exactly our predictions!); `(n, 2).T @ (n, 2)` → `(2, 2)` table of column-to-column dots.

Useful dots: squared length $\|a\|^2 = a \cdot a$, length $\|a\| = \sqrt{a \cdot a}$, cosine of the angle $\cos = (a \cdot b)/(\|a\|\|b\|)$.
Seminar 2 connection: every sum in the system of equations is a dot — $\sum x_1 x_2$ = `x1.dot(x2)`, $\sum x_1 y$ = `x1.dot(y)`.

In [ ]:
x1, x2 = X[:, 0], X[:, 1]
print("elementwise x1 * x2 =", x1 * x2)
print("dot x1 . x2 =", x1.dot(x2), " == sum:", np.sum(x1 * x2))
print("squared length x1.x1 =", x1.dot(x1))
print("length |x1| =", np.sqrt(x1.dot(x1)), " == linalg.norm:", np.linalg.norm(x1))
cos12 = x1.dot(x2) / (np.linalg.norm(x1) * np.linalg.norm(x2))
print("cos(x1, x2) =", round(float(cos12), 4))

# Predictions as dots: each row dotted with w, plus b (broadcasting!)
w = np.array([2.0, 0.1])
print("X @ w =", X @ w, " (one value per row)")
print("column dots X.T @ X =\n", X.T @ X, " (2x2 table: s11, s12 / s12, s22)")

In [ ]:
# ✏️ TODO 6: predict() + mse() with dots — copy these into Seminar 2.
def predict(X, w, b):
    # YOUR CODE HERE: each row dotted with w, plus b (hint: X @ w + b)
    return X @ w + b

def mse(y_true, y_pred):
    # YOUR CODE HERE: mean of squared misses
    err = y_pred - y_true
    return err.dot(err) / err.size  # == np.mean(err**2)

def cosine(a, b):
    # YOUR CODE HERE
    return a.dot(b) / (np.linalg.norm(a) * np.linalg.norm(b))

p = predict(X, np.array([2.0, 0.1]), 1.0)
assert np.allclose(p, [4.0, 7.0, 10.0, 13.0]), "check predict"
assert np.isclose(mse(y, p), 7.5), "check mse"
assert np.isclose(cosine(x1, x1), 1.0), "angle with itself is 0, cos = 1"
# every Seminar-2 sum is a dot:
assert np.isclose(x1.dot(x2), np.sum(x1 * x2)) and np.isclose(x1.dot(y), 70.0)
print("Task 6 OK ✓  (predict/mse are Seminar-2-ready)")

## 7. Random numbers (with seed = reproducible)

`np.random.default_rng(seed)` creates a generator; `uniform` and `normal` draw samples. Same seed → same numbers → your neighbour reproduces your result.

In [ ]:
rng = np.random.default_rng(42)
print("uniform 0..10:", rng.uniform(0, 10, size=5))
print("normal noise:", rng.normal(0, 1, size=5))

# ✏️ Task 7: re-run this cell — numbers do NOT change. Change seed to 0 and re-run — they do. Why does that matter for homework?

## 8. Plotting straight from NumPy arrays

`plt.scatter(x, y)` needs two flat arrays of shape `(n,)` — exactly what `X[:, 0]` and `y` are. No tables needed.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(X[:, 0], y)
axes[0].set_xlabel("x1"); axes[0].set_ylabel("y"); axes[0].set_title("y vs x1")
axes[1].scatter(X[:, 1], y, color="darkorange")
axes[1].set_xlabel("x2"); axes[1].set_ylabel("y"); axes[1].set_title("y vs x2")
plt.tight_layout(); plt.show()

In [ ]:
# ✏️ TODO 8: predicted-vs-true plot with the "ideal" dashed diagonal.
yp = predict(X, np.array([2.0, 0.1]), 1.0)
plt.figure(figsize=(5, 5))
plt.scatter(yp, y)  # <-- YOUR CODE HERE (predicted on x-axis, true on y-axis)
mn, mx = min(y.min(), yp.min()), max(y.max(), yp.max())
plt.plot([mn, mx], [mn, mx], "r--", label="ideal")  # perfect predictions lie on this line
plt.xlabel("predicted"); plt.ylabel("true"); plt.title("Predicted vs True"); plt.legend()
plt.show()
# QUESTION (comment): are points above or below the diagonal? What does that mean?

## 9. Mini-project: full pipeline, NumPy only

Same recipe as Seminar 2, tiny version:
1. generate $x_1, x_2$ and $y = 4 + 2.5 x_1 - 1.8 x_2 + \text{noise}$;
2. predict with a guess $(b, w_1, w_2)$ using a dot product;
3. score with MSE (a dot divided by $n$); summarise the data with NumPy stats + correlation.

In [ ]:
# ✏️ TODO 9: finish the pipeline (NumPy only — no tables).
rng = np.random.default_rng(7)
Xs = rng.uniform(0, 10, size=(100, 2))
ys = 4.0 + 2.5 * Xs[:, 0] - 1.8 * Xs[:, 1] + rng.normal(0, 2.0, size=100)  # <-- YOUR CODE HERE (formula above)

# NumPy-only data portrait (mean/std/min/max + corrcoef)
print("mean:", np.mean(Xs, axis=0), np.mean(ys))
print("std :", np.std(Xs, axis=0), np.std(ys))
print("min :", np.min(Xs, axis=0), np.min(ys))
print("max :", np.max(Xs, axis=0), np.max(ys))
print("corr(x1, x2):", round(float(np.corrcoef(Xs[:, 0], Xs[:, 1])[0, 1]), 3))

# Try the TRUE weights — MSE should be ≈ noise variance (sigma^2 = 4)
p_true = predict(Xs, np.array([2.5, -1.8]), 4.0)
print("MSE with true weights:", round(float(mse(ys, p_true)), 3), "(expect ≈ 4)")

# Try zeros — much worse
p_zero = predict(Xs, np.array([0.0, 0.0]), 0.0)
print("MSE with zeros:", round(float(mse(ys, p_zero)), 3))
assert mse(ys, p_true) < mse(ys, p_zero), "true weights must beat zeros"
print("Mini-project OK ✓")

## 10. Final checklist ✅

You are ready for Seminar 2 if you can:
- [ ] slice rows/columns (`X[:, 0]`, `X[1:3]`, `X[::2]`), explain `(n,)` vs `(n, 1)`, and avoid the view-modifies-original bug with `.copy()`;
- [ ] filter rows with masks (`X[X[:, 1] >= 30]`) and find positions with `where` / `argmax`;
- [ ] reduce along axes (`sum` / `mean` with `axis=0 / axis=1`, `keepdims`);
- [ ] broadcast (`(X - mu) / sigma` with `mu` of shape `(2,)`) and predict shape results;
- [ ] tell `*` (elementwise) from `dot` / `@` (sum of products), compute a norm and a cosine;
- [ ] write `predict` (`X @ w + b`) and `mse` (dot divided by $n$) from memory.

Keep this notebook open next to Seminar 2 — copy `predict` and `mse` from Task 6 when you need them.